In [323]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [324]:
data = pd.read_csv('Marketingcampaigns.csv')
data['Location'].replace(['Brisbane', 'Sydney', 'Perth', 'Auckland'], [1, 2, 3,4], inplace=True)
data = data.fillna(data.mean())


C:\Users\Hp\AppData\Local\Temp\ipykernel_25152\1661993627.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Location'].replace(['Brisbane', 'Sydney', 'Perth', 'Auckland'], [1, 2, 3,4], inplace=True)
C:\Users\Hp\AppData\Local\Temp\ipykernel_25152\1661993627.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downca

In [325]:
data

,Customer id,Age,Gender,Location,Email Opened,Email Clicked,Product page visit,Discount offered,Purchased
0,1,22,0,3,1,1,3,1,1
1,2,55,0,4,1,0,0,0,0
2,3,15,1,2,0,1,2,1,1
3,4,25,0,1,1,1,5,1,0
4,5,36,1,1,0,1,1,1,0
5,6,30,0,2,0,0,0,0,0
6,7,28,1,2,0,0,3,1,1
7,8,19,1,2,1,1,2,0,0
8,9,59,0,3,1,1,1,0,0
9,10,45,1,4,0,0,0,0,0


In [326]:
y_train = data['Purchased']
X_train = data.drop(columns=['Purchased'])


In [327]:
def regularization(m, weights, lambda_):  ## update yapılırken eklenecek regularization fonksiyonu
    reg = (lambda_ / (m)) * weights
    return reg


In [328]:
def scale_data(train, test = None):
    train = (train - train.mean()) / train.std()
    if test is not None:
        test = (test - train.mean()) / train.std()
        return train, test
    return train
   

In [329]:
def sigmoid(w, b, X_train):
    z = np.dot(X_train, w) + b
    y_pred=  1 / (1 + np.exp(-z))
    return np.clip(y_pred, 1e-15, 1 - 1e-15)

In [330]:
def update_weights(X_train, y_train, weights, bias, learning_rate, lambda_):
    m = X_train.shape[0]
    y_pred = sigmoid(weights, bias, X_train)
    error = y_pred - y_train
    dw = (1/m) * np.dot(X_train.T, error) + regularization(X_train.shape[0], weights, lambda_)
    db = (1/m)* np.sum(error)
    weights -= learning_rate * dw
    bias -= learning_rate * db
    return weights, bias

In [331]:
def cost_function(loss):
    m= loss.shape[0]
    return 1/m * np.sum(loss)

In [332]:
def train_model(X_train, y_train, weights, bias, learning_rate, epochs, lambda_):
    m = X_train.shape[0]
    n = X_train.shape[1]
    for epoch in range(epochs):
        weights, bias = update_weights(X_train, y_train, weights, bias, learning_rate, lambda_)
        if epoch % 1000 == 0:
            y_pred = sigmoid(weights, bias, X_train)
            loss = ((-y_train * np.log(y_pred))+ ((1-y_train)*(-1*np.log(1-y_pred))))
            print(f'Epoch: {epoch}, Cost: {cost_function(loss):.2f}')
    return weights, bias


    

In [333]:
def predict(X_test, weights_final, bias_final, y_test):
    m= X_test.shape[0]
    predictions = sigmoid(weights_final, bias_final, X_test)
    predictions = [1 if p>=0.5 else 0 for p in predictions]
    for i in range(m):
        print(f"Prediction:{predictions[i]} Real value: {y_test.iloc[i]}")
    return predictions


In [334]:
def accuracy(y_true, y_pred):
    correct = sum(y_true == y_pred)
    return correct / len(y_true) * 100

In [335]:
test_data = pd.read_csv('test_verisi.csv')
test_data['Location'].replace(['Brisbane', 'Sydney', 'Perth', 'Auckland'], [1, 2, 3,4], inplace=True)
test_data = test_data.fillna(test_data.mean())
X_test = test_data.drop(columns=['Purchased'])
Y_test = test_data['Purchased']
X_train, X_test = scale_data(X_train,X_test)


C:\Users\Hp\AppData\Local\Temp\ipykernel_25152\4213007699.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_data['Location'].replace(['Brisbane', 'Sydney', 'Perth', 'Auckland'], [1, 2, 3,4], inplace=True)
C:\Users\Hp\AppData\Local\Temp\ipykernel_25152\4213007699.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_d

In [342]:

weights = np.zeros(X_train.shape[1])
bias = 0
learning_rate = 0.1
epochs = 10000
lambda_ = 0.01
weights_final, bias_final = train_model(X_train, y_train, weights, bias, learning_rate, epochs, lambda_)
y_pred = predict(X_test, weights_final, bias_final, Y_test)
acc = accuracy(Y_test, y_pred)
print(f'Accuracy: {acc:.2f}%')

Epoch: 0, Cost: 0.68
Epoch: 1000, Cost: 0.28
Epoch: 2000, Cost: 0.25
Epoch: 3000, Cost: 0.24
Epoch: 4000, Cost: 0.23
Epoch: 5000, Cost: 0.22
Epoch: 6000, Cost: 0.21
Epoch: 7000, Cost: 0.21
Epoch: 8000, Cost: 0.20
Epoch: 9000, Cost: 0.20
Prediction:0 Real value: 0
Prediction:1 Real value: 1
Prediction:1 Real value: 1
Prediction:1 Real value: 1
Prediction:1 Real value: 0
Accuracy: 80.00%
